<a href="https://colab.research.google.com/github/memusleh-lab/aeco-ppe-yolov8/blob/main/notebooks/02_train_eval_FMP_Group3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# YOLOv8 Object Detection Project  
## M4U3 Final Major Project – Group 3

---

## 1. AECO Problem Framing

This project addresses a construction site safety use case: automated detection of hard hats (PPE compliance) using computer vision. The objective is to support safety supervisors in identifying potential PPE violations from site imagery.

In safety-critical applications, **false negatives are more dangerous than false positives**, as missing a worker without a hard hat may lead to unaddressed hazards. Therefore, the model prioritizes high recall.

---

## 2. Dataset Overview

The dataset is hosted on Roboflow and exported in YOLOv8 format with an 80/20 split:

- 80% Training set
- 20% Validation set
- Test set = 0% (validation used for evaluation)

Classes:
- hard_hat
- no_hard_hat

The dataset version is documented in the README for reproducibility.

---

## 3. Project Objectives

This notebook performs:

- Dataset download via Roboflow API
- YOLOv8 training (≥30 epochs)
- Evaluation using:
  - Precision
  - Recall
  - mAP50
  - mAP50–95
- Saving of training curves
- Inference on:
  - Validation images
  - New unseen images
- Reproducibility proof (environment + configuration)

---

## 4. Reproducibility Statement

This notebook is designed to run end-to-end in Google Colab without local installation.  
All dependencies are installed within the notebook and dataset access is handled via API.

To reproduce:
1. Enable GPU.
2. Run all cells from top to bottom.
3. Metrics, curves, and inference outputs will be generated automatically.

## A. Runtime & GPU Check

This cell verifies whether Google Colab is running with GPU acceleration.  
Training YOLOv8 for 30+ epochs is significantly faster on GPU. The GPU name is printed for the reproducibility proof.

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

CUDA available: True
GPU name: Tesla T4


## B. Environment Setup & Output Directories

This section installs required libraries (Ultralytics YOLOv8 + Roboflow) and prints environment details (Python, Torch, CUDA, Ultralytics version) for reproducibility.

It also creates a consistent output structure:
- `results/` for training runs and predictions  
- `results/curves/` for saved plots and curves  
- `results/evidence/` for evidence screenshots and prediction samples  

In [ ]:
!pip -q install ultralytics==8.* roboflow

import sys, torch, platform
from pathlib import Path

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

ROOT = Path.cwd()
RESULTS_DIR = ROOT / "results"
CURVES_DIR = RESULTS_DIR / "curves"
EVIDENCE_DIR = RESULTS_DIR / "evidence"
for p in [RESULTS_DIR, CURVES_DIR, EVIDENCE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

!pip show ultralytics | sed -n '1,5p'
print("ROOT:", ROOT)
print("RESULTS_DIR:", RESULTS_DIR)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 38.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.6/94.6 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 90.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 148.0 MB/s eta 0:00:00
Python: 3.12.12
Platform: Linux-6.6.113+-x86_64-with-glibc2.35
Torch: 2.10.0+cu128
CUDA: True
Name: ultralytics
Version: 8.4.19
Summary: Ultralytics YOLO 🚀 for SOTA object detection, multi-object tracking, instance segmentation, pose estimation and image classification.
Home-page: https://ultralytics.com
Author: 
ROOT: /content
RESULTS_DIR: /content/results


## C. Dataset Download (Roboflow – YOLOv8 Export)

The dataset is downloaded from Roboflow in YOLOv8 format using a fixed dataset version.  
This supports reproducibility because the dataset source and version remain consistent.

**Note:** API keys should not be hardcoded in public notebooks. Use Colab Secrets or a prompt input method.

In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="w36YxNlsftoXK1pSr57D")
project = rf.workspace("zigurat-assignment").project("ppe-hardhat-detection")
version = project.version(2)
dataset = version.download("yolov8")


loading Roboflow workspace...
loading Roboflow project...


In [ ]:
from roboflow import Roboflow
from pathlib import Path

RF_API_KEY = "w36YxNlsftoXK1pSr57D"
WORKSPACE = "zigurat-assignment"
PROJECT = "ppe-hardhat-detection"
VERSION = 2  # changed to your dataset version

rf = Roboflow(api_key=RF_API_KEY)
project = rf.workspace(WORKSPACE).project(PROJECT)
dataset = project.version(VERSION).download("yolov8")

DATASET_DIR = Path(dataset.location)
DATA_YAML = DATASET_DIR / "data.yaml"

print("Dataset dir:", DATASET_DIR)
print("Data yaml exists:", DATA_YAML.exists())

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to PPE-HardHat-Detection-2 in yolov8:: 100%|██████████| 10008/10008 [00:01<00:00, 5570.00it/s]

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Dataset dir: /content/PPE-HardHat-Detection-2
Data yaml exists: True


## D. **Optional**: Quick Verification Training (5 epochs)

This optional step performs a short 5-epoch training run to quickly validate that the pipeline works end-to-end (dataset path, configuration, Ultralytics installation).

The official training run for the assignment is executed in the next section with ≥30 epochs.

In [ ]:
from ultralytics import YOLO

MODEL_VARIANT = "yolov8n.pt"  # change to yolov8s.pt if you trained s
VERIFY_EPOCHS = 5
BATCH = 16
IMGSZ = 640

model = YOLO(MODEL_VARIANT)
model.train(
    data=str(DATA_YAML),
    epochs=VERIFY_EPOCHS,
    batch=BATCH,
    imgsz=IMGSZ,
    project=str(RESULTS_DIR),
    name="verify_train",
    exist_ok=True
)

print("Verification training complete.")

Ultralytics 8.4.19 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/PPE-HardHat-Detection-2/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=5, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=verify_train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patienc

## E. Full Training Run (≥30 epochs)

This is the main training run required by the assignment.  
The model is trained for **30+ epochs** on the Roboflow YOLOv8 dataset.

Training parameters (model variant, epochs, batch size, image size) are documented for reproducibility and must also be reflected in the README.

In [ ]:
from ultralytics import YOLO
from pathlib import Path

# --- Training configuration (document these in README) ---
MODEL_VARIANT = "yolov8n.pt"   # change to yolov8s.pt if you trained s
EPOCHS = 30                    # >=30 required
BATCH = 16                     # adjust if you get out-of-memory
IMGSZ = 640                    # common YOLO default

RESULTS_DIR = Path("results")
RUN_NAME = "train_30ep"

model = YOLO(MODEL_VARIANT)

train_results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    project=str(RESULTS_DIR),
    name=RUN_NAME,
    exist_ok=True
)

print("Training complete.")
print("Run directory:", RESULTS_DIR / RUN_NAME)

Ultralytics 8.4.19 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/PPE-HardHat-Detection-2/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train_30ep, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience

## F. Locate Trained Weights (best.pt)

After training, Ultralytics saves the best performing model weights at:
`results/<run_name>/weights/best.pt`

This cell confirms the weights exist before evaluation and inference.

In [ ]:
from pathlib import Path

RUN_DIR = Path("results") / RUN_NAME
WEIGHTS_PATH = RUN_DIR / "weights" / "best.pt"

print("Best weights path:", WEIGHTS_PATH)
print("Weights exist:", WEIGHTS_PATH.exists())

Best weights path: results/train_30ep/weights/best.pt
Weights exist: False


## G. Training Curves & Metrics Plots

This section collects the generated training plots (loss curves, PR curve, confusion matrix, etc.) and copies them into:
`results/curves/`

These plots are part of the required training outputs evidence pack.

In [ ]:
import shutil
from pathlib import Path

CURVES_DIR = Path("results") / "curves"
CURVES_DIR.mkdir(parents=True, exist_ok=True)

RUN_DIR = Path("results") / RUN_NAME

# Common Ultralytics plot filenames (some may not exist depending on version)
PLOTS = [
    "results.png",
    "confusion_matrix.png",
    "confusion_matrix_normalized.png",
    "PR_curve.png",
    "F1_curve.png",
    "P_curve.png",
    "R_curve.png",
]

copied = []
for fname in PLOTS:
    src = RUN_DIR / fname
    if src.exists():
        shutil.copy(src, CURVES_DIR / fname)
        copied.append(fname)

print("Copied plots:", copied)
print("Curves saved to:", CURVES_DIR)

Copied plots: []
Curves saved to: results/curves


## H. Validation Evaluation (Metrics Table)

This section evaluates the trained model on the validation split and prints:
- Precision
- Recall
- mAP@0.5 (mAP50)
- mAP@0.5:0.95 (mAP50–95)

These metrics are required in the assignment deliverables and will be summarized in the README and report.

In [ ]:
from ultralytics import YOLO
from pathlib import Path

# Redefine RUN_NAME to ensure consistency within this cell if needed
RUN_NAME = "train_30ep"

# Construct the correct path for weights based on Ultralytics' default output structure
# Ultralytics saves runs under 'runs/detect/' within the specified project directory
ACTUAL_RUNS_ROOT = Path("runs") / "detect" / "results"
ACTUAL_RUN_DIR = ACTUAL_RUNS_ROOT / RUN_NAME
WEIGHTS_PATH = ACTUAL_RUN_DIR / "weights" / "best.pt"

best_model = YOLO(str(WEIGHTS_PATH))
metrics = best_model.val(data=str(DATA_YAML), imgsz=IMGSZ)

print("Precision:", metrics.box.p[0].item())
print("Recall:", metrics.box.r[0].item())
print("mAP50:", metrics.box.map50.item())
print("mAP50-95:", metrics.box.map.item())

Ultralytics 8.4.19 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,006,233 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1209.2±198.0 MB/s, size: 53.9 KB)
val: Scanning /content/PPE-HardHat-Detection-2/valid/labels.cache... 1000 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1000/1000 299.6Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 63/63 5.6it/s 11.3s
                   all       1000       5040       0.94      0.582      0.624      0.391
                  head        202       1229      0.898      0.836      0.892      0.545
                helmet        919       3686      0.922       0.91      0.958      0.616
                person         28        125          1          0     0.0236     0.0115
Speed: 1.6ms preprocess, 3.8ms inference, 0.0ms loss, 1.3ms postprocess per image
Results saved to /content/runs/dete

## I. Inference Evidence – Validation Predictions (10 samples)

This section runs inference on a sample of 10 validation images and saves prediction outputs to:
`results/val_preds/`

These images form part of the evidence pack required by the assignment.

In [ ]:
from pathlib import Path

VAL_IMG_DIR = DATASET_DIR / "valid" / "images"
val_imgs = sorted(list(VAL_IMG_DIR.glob("*")))

print("Validation images found:", len(val_imgs))
sample_val = val_imgs[:10]

best_model.predict(
    source=[str(p) for p in sample_val],
    save=True,
    project="results",
    name="val_preds",
    exist_ok=True
)

print("Saved validation predictions to:", Path("results") / "val_preds")

Validation images found: 1000

0: 640x640 4 helmets, 5.0ms
1: 640x640 3 heads, 5 helmets, 5.0ms
2: 640x640 6 helmets, 5.0ms
3: 640x640 3 helmets, 5.0ms
4: 640x640 4 helmets, 5.0ms
5: 640x640 6 heads, 5.0ms
6: 640x640 7 helmets, 5.0ms
7: 640x640 7 helmets, 5.0ms
8: 640x640 6 helmets, 5.0ms
9: 640x640 4 helmets, 5.0ms
Speed: 1.9ms preprocess, 5.0ms inference, 0.8ms postprocess per image at shape (1, 3, 640, 640)
Results saved to /content/runs/detect/results/val_preds
Saved validation predictions to: results/val_preds


## J. Inference Evidence – New Images (5 samples)

This section runs inference on 5 **new images** (not part of the dataset split) to demonstrate generalization.  
Place new images in the `new_images/` folder before running this cell.

Outputs are saved to:
`results/new_preds/`

In [ ]:
from pathlib import Path

NEW_IMG_DIR = Path("new_images")
NEW_IMG_DIR.mkdir(exist_ok=True)

new_imgs = sorted(list(NEW_IMG_DIR.glob("*")))
print("New images found:", len(new_imgs))

sample_new = new_imgs[:5]

best_model.predict(
    source=[str(p) for p in sample_new],
    save=True,
    project="results",
    name="new_preds",
    exist_ok=True
)

print("Saved new image predictions to:", Path("results") / "new_preds")

You can upload files from your local computer to the Colab environment using `google.colab.files.upload()`. This will open a file selection dialog in your browser. After uploading, we'll move these files into the `new_images` directory.

Alternatively, you can manually drag and drop files into the `new_images` folder using the file browser on the left-hand side of Colab (the folder icon).

In [ ]:
from google.colab import files
from pathlib import Path

NEW_IMG_DIR = Path("new_images")
NEW_IMG_DIR.mkdir(exist_ok=True)

print("Please select the images you want to upload to the new_images folder.")
uploaded_files = files.upload()

for filename in uploaded_files.keys():
    # Move the uploaded files to the new_images directory
    destination_path = NEW_IMG_DIR / filename
    Path(filename).rename(destination_path)
    print(f"Moved '{filename}' to '{destination_path}'")

print(f"Successfully added {len(uploaded_files)} images to '{NEW_IMG_DIR}'.")


Please select the images you want to upload to the new_images folder.


## K. Evidence Pack Organization

This step copies inference outputs into the structured evidence folders required by the repository:
- `results/evidence/val_preds/` (10 validation predictions)
- `results/evidence/new_preds/` (5 new-image predictions)

Annotation screenshots (3–5 examples) should be placed manually in:
`results/evidence/annotations/`

In [ ]:
import shutil
from pathlib import Path

EVIDENCE_BASE = Path("results") / "evidence"
VAL_EVID = EVIDENCE_BASE / "val_preds"
NEW_EVID = EVIDENCE_BASE / "new_preds"

VAL_EVID.mkdir(parents=True, exist_ok=True)
NEW_EVID.mkdir(parents=True, exist_ok=True)

# YOLO saves predicted images in: results/<name>/
VAL_PRED_DIR = Path("results") / "val_preds"
NEW_PRED_DIR = Path("results") / "new_preds"

# Copy predicted JPG/PNG outputs to evidence folders
def copy_preds(src_dir: Path, dst_dir: Path, limit: int):
    imgs = sorted([p for p in src_dir.glob("*") if p.suffix.lower() in [".jpg", ".jpeg", ".png"]])
    for p in imgs[:limit]:
        shutil.copy(p, dst_dir / p.name)
    return [p.name for p in imgs[:limit]]

val_copied = copy_preds(VAL_PRED_DIR, VAL_EVID, 10)
new_copied = copy_preds(NEW_PRED_DIR, NEW_EVID, 5)

print("Copied val evidence:", val_copied)
print("Copied new evidence:", new_copied)
print("Evidence base:", EVIDENCE_BASE)

## L. Reproducibility Proof

This section prints a reproducibility proof record including:
- Date/time (UTC)
- Platform + Ultralytics version
- GPU availability and GPU name
- Training parameters
- Run directory and weights path

Copy this output into the README under “Reproducibility Proof”.

In [ ]:
import time, platform, torch
from ultralytics import __version__ as ulty_ver

print("Reproducibility Proof")
print("- Date/Time (UTC):", time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime()))
print("- Platform:", platform.platform())
print("- Ultralytics:", ulty_ver)
print("- CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("- GPU:", torch.cuda.get_device_name(0))
print(f"- Train config: model={MODEL_VARIANT}, epochs={EPOCHS}, batch={BATCH}, imgsz={IMGSZ}")
print("- Run directory:", str(Path("results") / RUN_NAME))
print("- Best weights:", str(WEIGHTS_PATH))